In [9]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
CLC → Custom Caption Generator (center-weighted, simplified output)

Computes center-weighted captions (≥2% presence rule)
from grayscale CLC_reference tiles and produces compact, comma-separated
captions in lowercase.

Outputs:
  BaseFolder, BaseFilename, CLC_codes, Caption_5
"""

import os, re
import numpy as np
import pandas as pd
import rasterio
from IPython.display import clear_output

# ------------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------------
PATCH_DIR = "/home/ubuntu/SENSERO/GeoTiff/Patch_64"
OUT_PARQUET = os.path.join(PATCH_DIR, "captions_64_custom.parquet")
OUT_CSV     = os.path.join(PATCH_DIR, "captions_64_custom.csv")
# ------------------------------------------------------------------
# SENSERO CLASS MAPPING
# ------------------------------------------------------------------
ben_map = {
    111: "urban fabric", 112: "urban fabric",
    121: "urban fabric",
    122: "unlabeled", 123: "unlabeled", 124: "unlabeled",
    131: "quarries or dump sites", 132: "quarries or dump sites", 133: "quarries or dump sites",
    141: "unlabeled", 142: "unlabeled",
    211: "agriculture", 212: "agriculture", 213: "agriculture",
    221: "permanent crops", 222: "permanent crops", 223: "permanent crops",
    231: "pastures",
    241: "mixed farmland",
    242: "mixed farmland",
    243: "mixed farmland",
    244: "mixed farmland",
    311: "forests", 312: "forests", 313: "forests",
    321: "shrubland",
    322: "shrubland",
    323: "shrubland",
    324: "shrubland",
    331: "unlabeled",
    332: "unlabeled",
    333: "bare ground",
    334: "bare ground", 335: "bare ground",
    411: "wetlands", 412: "wetlands",
    421: "wetlands", 422: "wetlands",
    423: "wetlands",
    511: "water bodies", 512: "water bodies",
    521: "water bodies", 522: "water bodies", 523: "water bodies",
    999: "water bodies",
}
IGNORE_CODES = {c for c, n in ben_map.items() if n == "unlabeled"}
HUMAN_MADE = {"urban fabric", "quarries or dump sites"}

# ------------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------------
def extract_base_filename(path: str) -> str:
    """Return patch core name: no extension or suffix (_CLC, _Bxxx, etc.)."""
    name = os.path.splitext(os.path.basename(path))[0]
    name = re.sub(r"_CLC.*$", "", name)
    name = re.sub(r"_B\d+$", "", name)
    return name

def reorder_center_weighted(dominant, center_hist, min_center_pct=1.0):
    """Prioritize human-made → other center → remaining."""
    center_hm = [n for n, p in center_hist.items() if n in HUMAN_MADE and p >= min_center_pct]
    center_oth = [n for n, p in center_hist.items() if n not in HUMAN_MADE and p >= min_center_pct]
    center_hm.sort(key=lambda n: -center_hist[n])
    center_oth.sort(key=lambda n: -center_hist[n])

    center_present = set(center_hm + center_oth)
    tail = [name for name, _, _ in dominant if name not in center_present]
    order = center_hm + center_oth + tail
    dmap = {name: (name, cnt, pct) for name, cnt, pct in dominant}
    return [dmap[n] for n in order if n in dmap]

# ------------------------------------------------------------------
# ANALYSIS
# ------------------------------------------------------------------
def analyze_clc_tif_bigearthnet(path, center_frac=0.2):
    """Compute center-weighted caption and class list."""
    with rasterio.open(path) as src:
        clc = src.read(1).astype(np.int32)

    mask = clc > 0
    if not np.any(mask):
        return "unknown land cover", []

    flat = clc[mask].flatten()

    # map codes → names, filter unlabeled
    names = np.array([ben_map.get(int(c), "unlabeled") for c in flat], dtype=object)
    valid = names != "unlabeled"
    if not np.any(valid):
        return "unknown land cover", []

    flat = flat[valid]
    names = names[valid]

    # global stats by numeric code
    u_codes, counts = np.unique(flat, return_counts=True)
    total = counts.sum()
    keep_idx = [i for i in np.argsort(-counts) if 100 * counts[i] / total >= 2.0]
    kept_codes = [int(u_codes[i]) for i in keep_idx]
    if not kept_codes:
        return "unknown land cover", []

    # dominant list by mapped NAME
    kept_mask = np.isin(flat, kept_codes)
    kept_names = names[kept_mask]
    u_names, n_counts = np.unique(kept_names, return_counts=True)
    n_total = n_counts.sum()
    dominant = [(str(u_names[i]), int(n_counts[i]), 100 * n_counts[i] / n_total)
                for i in np.argsort(-n_counts)]

    # center weighting
    h, w = clc.shape
    cside = int(center_frac * min(h, w))
    y0, y1 = h // 2 - cside // 2, h // 2 + cside // 2
    x0, x1 = w // 2 - cside // 2, w // 2 + cside // 2
    mask_center = np.zeros((h, w), dtype=bool)
    mask_center[y0:y1, x0:x1] = True

    center_flat = clc[mask & mask_center].flatten()
    center_names = np.array([ben_map.get(int(c), "unlabeled") for c in center_flat], dtype=object)
    valid_center = (center_names != "unlabeled") & np.isin(center_flat, kept_codes)
    center_names = center_names[valid_center]

    center_hist = {}
    if center_names.size > 0:
        cu, cc = np.unique(center_names, return_counts=True)
        ctot = cc.sum()
        for n, cnt in zip(cu, cc):
            center_hist[str(n)] = 100 * cnt / ctot

    dominant = reorder_center_weighted(dominant, center_hist)

    # caption
    ordered_names = [n.replace(",", "").strip().lower() for n, _, _ in dominant]
    ordered_names = list(dict.fromkeys(n for n in ordered_names if n))
    caption = ", ".join(ordered_names) if ordered_names else "unknown land cover"

    return caption, kept_codes

# ------------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------------
def main():
    if os.path.exists(OUT_PARQUET):
        df = pd.read_parquet(OUT_PARQUET)
        df = df.drop_duplicates(subset=["BaseFilename"])
        done = set(df["BaseFilename"])
    else:
        df = pd.DataFrame(columns=["BaseFolder", "BaseFilename", "CLC_codes", "Caption_5"])
        done = set()

    clc_root = os.path.join(PATCH_DIR, "CLC_reference")

    for base in sorted(os.listdir(clc_root)):
        base_dir = os.path.join(clc_root, base)
        if not os.path.isdir(base_dir):
            continue

        for f in sorted(os.listdir(base_dir)):
            if not f.lower().endswith(".tif"):
                continue

            clc_path = os.path.join(base_dir, f)
            base_fn = extract_base_filename(clc_path)
            if base_fn in done:
                continue

            try:
                caption, codes = analyze_clc_tif_bigearthnet(clc_path)
            except Exception as e:
                print(f"[WARN] {f}: {e}")
                caption, codes = "unknown land cover", []

            row = {
                "BaseFolder": base,
                "BaseFilename": base_fn,
                "CLC_codes": ", ".join(map(str, codes)),
                "Caption_5": caption
            }

            df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
            df = df.drop_duplicates(subset=["BaseFilename"])
            df.to_parquet(OUT_PARQUET, index=False)
            clear_output(wait=False)
            print(f"[INFO] {len(df)} patches → {caption}")

    #df.to_csv(OUT_CSV, index=False)
    #print(f"[DONE] {len(df)} BigEarthNet captions saved\n  CSV: {OUT_CSV}\n  Parquet: {OUT_PARQUET}")

if __name__ == "__main__":
    main()


[INFO] 10000 patches → pastures, agriculture
